In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [3]:
from langgraph.store.memory import InMemoryStore

# 1. Store 초기화
store = InMemoryStore()

user_id = "user_001"
application_context = "personal_assistant"

# 2. Namespace(폴더 경로) 설정
namespace = (user_id, application_context)


In [4]:
# 3. 첫 번째 데이터 저장
store.put(
    namespace,
    "memory_001",  # 이 데이터의 고유 식별자 (Key)
    {
        "facts": [
            "사용자는 커피보다 차를 선호함",
            "사용자는 매일 아침 6시에 일어남",
        ],
        "language": "Korean",
    },
)


In [5]:
# 4. 두 번째 데이터 저장
store.put(
    namespace,
    "memory_002",  # 이전 데이터가 덮어씌워지지 않도록 새로운 Key 지정
    {
        "facts": [
            "사용자는 그림 회화 작품을 좋아함",
            "빈센트 반 고흐의 작품을 특히 좋아함",
        ]
    },
)


In [6]:
# 특정 Key 조회 (get)
item1 = store.get(namespace, "memory_001")
# print(item1)
print("1번 메모리:", item1.value)

item2 = store.get(namespace, "memory_002")
print("2번 메모리:", item2.value)

# Namespace 전체 검색 (search)
print("\n--- 전체 검색 결과 ---")
items = store.search(namespace)
# print(items)
for item in items:
    print(f"Key: {item.key} | Value: {item.value}")


1번 메모리: {'facts': ['사용자는 커피보다 차를 선호함', '사용자는 매일 아침 6시에 일어남'], 'language': 'Korean'}
2번 메모리: {'facts': ['사용자는 그림 회화 작품을 좋아함', '빈센트 반 고흐의 작품을 특히 좋아함']}

--- 전체 검색 결과 ---
Key: memory_001 | Value: {'facts': ['사용자는 커피보다 차를 선호함', '사용자는 매일 아침 6시에 일어남'], 'language': 'Korean'}
Key: memory_002 | Value: {'facts': ['사용자는 그림 회화 작품을 좋아함', '빈센트 반 고흐의 작품을 특히 좋아함']}


In [7]:
from dataclasses import dataclass

@dataclass
class Context:
    user_id: str
    app_name: str


In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable


@wrap_model_call
def inject_memory(request: ModelRequest, handler: Callable) -> ModelResponse:
    current_user = request.runtime.context.user_id
    current_app = request.runtime.context.app_name
    memories = request.runtime.store.search((current_user, current_app))

    memory_content = "기록된 정보 없음"

    if memories:
        # 검색된 메모리들을 텍스트로 변환
        extracted_facts = []
        for item in memories:
            # print('item', item)
            if "facts" in item.value:
                extracted_facts.extend(item.value["facts"])
                # print('extracted_facts', extracted_facts)
        memory_content = "\n- ".join(extracted_facts)
        # print('memory_content', memory_content)

    system_message = f"사용자 관련 장기 메모리 :\n- {memory_content}"
    request = request.override(system_prompt=system_message)
    return handler(request)


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    store=store,          # store 연결
    context_schema=Context,
    middleware=[inject_memory]  # 미들웨어 장착
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고있는 정보 알려줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
    # user_id가 002로 바뀌면 사용자 정보를 알 수 없다고 답변한다 
)
print(response["messages"][-1].content)


item Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 커피보다 차를 선호함', '사용자는 매일 아침 6시에 일어남'], 'language': 'Korean'}, created_at='2026-03-22T10:33:05.569952+00:00', updated_at='2026-03-22T10:33:05.569953+00:00', score=None)
extracted_facts ['사용자는 커피보다 차를 선호함', '사용자는 매일 아침 6시에 일어남']
item Item(namespace=['user_001', 'personal_assistant'], key='memory_002', value={'facts': ['사용자는 그림 회화 작품을 좋아함', '빈센트 반 고흐의 작품을 특히 좋아함']}, created_at='2026-03-22T10:33:06.793430+00:00', updated_at='2026-03-22T10:33:06.793431+00:00', score=None)
extracted_facts ['사용자는 커피보다 차를 선호함', '사용자는 매일 아침 6시에 일어남', '사용자는 그림 회화 작품을 좋아함', '빈센트 반 고흐의 작품을 특히 좋아함']
memory_content 사용자는 커피보다 차를 선호함
- 사용자는 매일 아침 6시에 일어남
- 사용자는 그림 회화 작품을 좋아함
- 빈센트 반 고흐의 작품을 특히 좋아함
다음이 현재 제가 기억하고 있는 당신에 관한 정보입니다.

- 커피보다 차를 선호합니다.
- 매일 아침 6시에 일어납니다.
- 그림 회화 작품을 좋아합니다.
- 특히 빈센트 반 고흐의 작품을 좋아합니다.

원하신다면 이 정보를 바탕으로 개인화된 제안이나 대화를 더 자세하게 맞춰드릴 수 있어요. 예를 들어 차 추천, 6시 기상 루틴 아이디어, 반 고흐 테마의 미술관 전시 소식이나 작품 해설 등도 제공해 드릴 수 